[{'run_time': '2026-06-28T06:50:00', 'load_type': 'FULL', 'start_date': '2026-03-30', 'end_date': '2026-06-28', 'rows_loaded': 120000, 'files_created': 6, 'status': 'SUCCESS', 'duration_seconds': 45.2}, {'run_time': '2026-06-28T06:50:00', 'load_type': 'FULL', 'start_date': '2026-03-30', 'end_date': '2026-06-28', 'rows_loaded': 240000, 'files_created': 6, 'status': 'SUCCESS', 'duration_seconds': 45.2}]
[{'run_time': '2026-06-28T06:50:00', 'load_type': 'FULL', 'start_date': '2026-03-30', 'end_date': '2026-06-28', 'rows_loaded': 120000, 'files_created': 6, 'status': 'SUCCESS', 'duration_seconds': 45.2}, {'run_time': '2026-06-28T06:50:00', 'load_type': 'FULL', 'start_date': '2026-03-30', 'end_date': '2026-06-28', 'rows_loaded': 240000, 'files_created': 6, 'status': 'SUCCESS', 'duration_seconds': 45.2}, {'run_time': '2026-06-28T06:50:00', 'load_type': 'FULL', 'start_date': '2026-03-30', 'end_date': '2026-06-28', 'rows_loaded': 240000, 'files_created': 6, 'status': 'SUCCESS', 'duration_secon

In [6]:
state["last_created_date"] = "2026-06-28"
state["last_unique_key"] = 234567899
state["last_ingest_time"] = datetime.now().isoformat()

with open("current_state.json", "w") as f:
    json.dump(state, f, indent=4)

print(state)

def extract_data():
    base_url = "https://data.cityofnewyork.us/api/odata/v4/erm2-nwe9"
    date_cutoff = datetime.now() - timedelta(days=90)
    date_str = date_cutoff.strftime("%Y-%m-%d")
    url = f"{base_url}?$filter=created_date ge {date_str}&$format=json"
    

In [3]:
date_cutoff = datetime.now() - timedelta(days=90)
date_str = date_cutoff.strftime("%Y-%m-%dT00:00:00.000")

In [ ]:
base_url = "https://data.cityofnewyork.us/api/odata/v4/erm2-nwe9"
date_cutoff = datetime.now() - timedelta(days=90)
date_str = date_cutoff.strftime("%Y-%m-%dT00:00:00.000")
url = (
    "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
    f"?$select=count(*)&$where=created_date >= '{date_str}'"
)


all_data = []

while url:
    response = requests.get(url)
    data = response.json()

    all_data.extend(data["value"])
    url = data.get("@odata.nextLink")

df = pd.DataFrame(all_data)

print(df.len())
df.head()

In [ ]:
all_data = []

while url:
    response = requests.get(url)
    data = response.json()

    all_data.extend(data["value"])
    url = data.get("@odata.nextLink")

df = pd.DataFrame(all_data)

df.head() 


In [ ]:
import pyarrow 
import boto3
import s3fs
import awscli



In [ ]:
s3 = boto3.client('s3')
response = s3.list_buckets()
for i in response['Buckets']:
    print(i['Name'])

nyi-data-analytics


In [3]:
s3.upload_file(
    'chcek.csv', #file in our system
    'nyi-data-analytics' , #bucket name
    'first_thousand'
)

In [3]:
import pyarrow 
import boto3
import s3fs
import awscli
from io import BytesIO
from datetime import datetime , timedelta
from zoneinfo import ZoneInfo


s3 = boto3.client('s3')
response = s3.list_buckets()
#print(Bucket Name)
for i in response['Buckets']:
    print(i['Name'])



nyi-data-analytics


In [7]:
def generate_folder_name() :
    timestamp = datetime.now()
    ts= timestamp.strftime("%Y%m%d_%H%M%S")
    return f"ingest_date_{ts}"

folder_name = generate_folder_name()
print(folder_name)

ingest_date_20260701_092624


In [19]:
def upload_df_to_s3(df, bucket_name, base_prefix, folder_name, file_name):
    
    buffer = BytesIO()

    #dataframe of raw_data
    #converts raw data to parquet and stores it in ram instead of Disk
    df.to_parquet(
        buffer,
        engine="pyarrow",
        index=False
    )
    
    #to bring cursor on start
    buffer.seek(0)

    #to create file path = object key
    key = f"{base_prefix}/{folder_name}/{file_name}.parquet"

    
    s3.upload_fileobj(
        buffer,
        bucket_name,
        key
    )

    return f"s3://{bucket_name}/{key}"

def generate_folder_name() :
    timestamp = datetime.now(ZoneInfo("Asia/Kolkata"))
    ts= timestamp.strftime("%Y%m%d_%H%M%S")
    return f"ingest_date_{ts}"

folder_name = generate_folder_name()
file_load = 1
s3 = boto3.client('s3')

df = pd.read_csv("chcek.csv")
upload_df_to_s3(df, 
                bucket_name = 'nyi-data-analytics',
                base_prefix = 'data/full_load',
                folder_name = folder_name,
                file_name = f"part_{file_load:03d}")
                

's3://nyi-data-analytics/data/full_load/ingest_date_20260701_095542/part_001.parquet'

In [10]:
import pandas as pd
df=pd.read_csv("chcek.csv")

In [17]:
buffer = BytesIO()
df.to_parquet(
        buffer,
        engine="pyarrow",
        index=False
    )

#to bring cursor on start
buffer.seek(0)

    key = f"{base_prefix}/{folder_name}/{file_name}.parquet"

    s3_client.upload_fileobj(
        buffer,
        bucket_name,
        key
    )

    return f"s3://{bucket_name}/{key}"


,Unnamed: 0,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,...,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,0,69396322,2026-06-18T02:06:17.000,NaN,NYPD,New York City Police Department,Illegal Parking,Double Parked Blocking Traffic,NaN,Street/Sidewalk,...,Car,NaN,NaN,NaN,NaN,NaN,NaN,40.644248,-73.880099,POINT (-73.88009914733 40.644247849899)
1,1,69390855,2026-06-18T02:05:50.000,NaN,DSNY,Department of Sanitation,Dirty Condition,Trash,Littering,Sidewalk,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.800151,-73.953215,POINT (-73.95321534259 40.800150933593)
2,2,69396370,2026-06-18T02:05:49.000,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Talking,NaN,Residential Building/House,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.646580,-73.950087,POINT (-73.950087024662 40.646580439376)
3,3,69397701,2026-06-18T02:05:36.000,NaN,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.693884,-74.000844,POINT (-74.000843832356 40.693883764345)
4,4,69400344,2026-06-18T02:05:24.000,NaN,NYPD,New York City Police Department,Drug Activity,Use Indoor,NaN,Other,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.806681,-73.927130,POINT (-73.92713042937 40.806680823068)


In [21]:
df_3 = pd.read_parquet("part_001.parquet")
df_3.head()

,Unnamed: 0,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,...,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,0,69396322,2026-06-18T02:06:17.000,NaN,NYPD,New York City Police Department,Illegal Parking,Double Parked Blocking Traffic,NaN,Street/Sidewalk,...,Car,NaN,NaN,NaN,NaN,NaN,NaN,40.644248,-73.880099,POINT (-73.88009914733 40.644247849899)
1,1,69390855,2026-06-18T02:05:50.000,NaN,DSNY,Department of Sanitation,Dirty Condition,Trash,Littering,Sidewalk,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.800151,-73.953215,POINT (-73.95321534259 40.800150933593)
2,2,69396370,2026-06-18T02:05:49.000,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Talking,NaN,Residential Building/House,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.646580,-73.950087,POINT (-73.950087024662 40.646580439376)
3,3,69397701,2026-06-18T02:05:36.000,NaN,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.693884,-74.000844,POINT (-74.000843832356 40.693883764345)
4,4,69400344,2026-06-18T02:05:24.000,NaN,NYPD,New York City Police Department,Drug Activity,Use Indoor,NaN,Other,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.806681,-73.927130,POINT (-73.92713042937 40.806680823068)


1000